# Run the Nova Sales Assistant on Colab

1. **Runtime → Change runtime type → T4 GPU**
2. Run the cells in order.
3. Open the `trycloudflare.com` link that the last cell prints. It works while this notebook is running.

In [ ]:
!git clone https://github.com/YOUR_USERNAME/nova-sales-assistant.git
%cd nova-sales-assistant
!pip install -q -r requirements.txt

Start the API server in the background. The first start downloads and loads the model, which takes a few minutes.

In [ ]:
import subprocess, time, urllib.request

server = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT,
)

while True:
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/health")
        print("Server is ready")
        break
    except Exception:
        if server.poll() is not None:
            print(open("server.log").read()[-3000:])
            raise RuntimeError("The server stopped. See the log above.")
        time.sleep(10)

Open a public link with Cloudflare Tunnel (free, no account needed).

In [ ]:
import re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("Web app :", match.group())
        print("API docs:", match.group() + "/docs")
        break

Server log (for debugging):

In [ ]:
!tail -n 30 server.log